# SOME/IP Anomaly Detection — Reproduction Pipeline

End-to-end pipeline reproducing and extending **Kim et al. (2026)**:
> Kim et al. *XGBoost-Based Anomaly Detection Framework for SOME/IP in In-Vehicle Networks.*  
> Systems, 14(2), 196. https://doi.org/10.3390/systems14020196

**Dataset**: 7 PCAP files from Figshare article 30970450  
https://figshare.com/articles/dataset/SOME_IP_traffic_normal_and_abnormal_traffic_/30970450

## Steps
| # | Script | Output | Time |
|---|--------|--------|------|
| 0 | `00_download.py` | `data/raw/*.pcap` (~2 GB) | 5–30 min |
| 1 | `01_parse.py` | `data/parsed/parsed_packets.csv` (~3.5 GB) | 30–60 min |
| 2 | `03_features.py` | `data/features/*.npy` | 30–60 min |
| 3 | `04_train.py` | `results/v2_proposed/model_binary.json` | 2–5 min |
| 4 | `05_evaluate.py` | Metrics printed | <1 min |

> **Note**: Steps 0–2 are I/O-heavy and are skipped automatically if outputs already exist.

In [ ]:
import subprocess, sys
from pathlib import Path

ROOT = Path().resolve().parent          # detection/
SRC  = ROOT / 'src'

def run(script, *args, cwd=ROOT):
    """Run a script and stream its output."""
    cmd = [sys.executable, str(SRC / script), *args]
    print(f'▶  {" ".join(str(c) for c in cmd)}\n')
    result = subprocess.run(cmd, cwd=str(cwd), capture_output=False, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'{script} exited with code {result.returncode}')

print(f'ROOT = {ROOT}')
print(f'Python = {sys.version}')

## Step 0 — Download PCAPs

Downloads 7 PCAP files from Figshare (~2 GB total) into `data/raw/`.  
Uses the Figshare public API — no account required.

In [ ]:
run('00_download.py')

## Step 1 — Parse PCAPs → CSV

Reads all 7 PCAPs with Scapy and extracts IP, TCP/UDP, and SOME/IP header fields.  
Labels each packet by source IP (known attacker addresses per PCAP).  
Output: `data/parsed/parsed_packets.csv` (~14.2 M rows, ~3.5 GB).

In [ ]:
run('01_parse.py')

## Step 2 — Feature Extraction

Extracts 18 features (f01–f18) per packet:

- **f01–f12**: Reproduction of Kim et al. (2026) — time intervals, byte-distribution likelihood/entropy, payload Hamming changes, length changes.
- **f13–f16**: Extensions — repeat rate, duplicate-source flag, SOME/IP and transport payload lengths.
- **f17**: `src_packet_rate` — packets/s from this source IP in a 1000-packet sliding window.
- **f18**: `src_payload_diversity` — unique payloads / total packets from this source IP (same window).

Output: `data/features/{X,y}_{train,test}.npy`, `train_features.csv`, `test_features.csv`.

In [ ]:
run('03_features.py')

## Step 3 — Train XGBoost (binary)

Trains a binary XGBoost classifier (normal vs attack).  
Saves model to `results/v2_proposed/model_binary.json`.

> **Why binary only?** The dataset labels attacks at PCAP level, not packet level —  
> background traffic in attack PCAPs is labeled with the attack type.  
> A multi-class approach conflates background normal traffic with attack classes.

In [ ]:
run('04_train.py', '--mode', 'binary')

## Step 4 — Evaluate

Loads the saved model and reports per-attack-type recall, plus feature importances.

In [ ]:
run('05_evaluate.py', '--model', 'binary')

## Results — Quick Analysis

Load the saved model and test data to produce plots.

In [ ]:
import sys
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc,
    precision_score, recall_score, f1_score,
)
from paths import FEATURES_DIR, RESULTS_V2

FEAT_COLS = [
    'f01_ip_time_interval', 'f02_someip_likelihood', 'f03_someipsd_likelihood',
    'f04_tcpudp_likelihood', 'f05_someip_entropy', 'f06_someipsd_entropy',
    'f07_tcpudp_entropy', 'f08_someip_payload_changes', 'f09_someipsd_payload_changes',
    'f10_tcpudp_payload_changes', 'f11_ip_length_changes', 'f12_tcpudp_length_changes',
    'f13_payload_repeat_rate', 'f14_duplicate_source',
    'f15_someip_payload_length', 'f16_tcpudp_payload_length',
    'f17_src_packet_rate', 'f18_src_payload_diversity',
]

X_test  = np.load(FEATURES_DIR / 'X_test.npy')
y_test  = np.load(FEATURES_DIR / 'y_test.npy').astype(int)
at_test = np.concatenate([
    c['attack_type'].values
    for c in pd.read_csv(FEATURES_DIR / 'test_features.csv',
                          usecols=['attack_type'], chunksize=500_000)
])

model = xgb.XGBClassifier()
model.load_model(str(RESULTS_V2 / 'model_binary.json'))

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f'F1    = {f1_score(y_test, y_pred):.4f}')
print(f'Prec  = {precision_score(y_test, y_pred):.4f}')
print(f'Rec   = {recall_score(y_test, y_pred):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Attack']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix')

# ── ROC curve ─────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, lw=1.5, label=f'AUC = {roc_auc:.4f}')
axes[1].plot([0,1],[0,1],'--', color='grey')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve'); axes[1].legend()

# ── Feature importances ───────────────────────────────────────────────────────
imps = model.feature_importances_
order = np.argsort(imps)[::-1]
top = 12
axes[2].barh([FEAT_COLS[i].split('_',1)[0] for i in order[:top]][::-1],
              imps[order[:top]][::-1])
axes[2].set_xlabel('Importance (gain)')
axes[2].set_title('Top-12 Features')

plt.tight_layout()
plt.savefig(ROOT / 'results' / 'v2_proposed' / 'overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-attack-type recall
attack_types = ['dos', 'fuzzy', 'mitm']
recalls = []
for at in attack_types:
    mask = (at_test == at)
    yt = y_test[mask]; yp = y_pred[mask]
    tp = ((yt==1) & (yp==1)).sum()
    fn = ((yt==1) & (yp==0)).sum()
    rc = tp / (tp+fn) if (tp+fn) > 0 else 0.0
    recalls.append(rc)
    print(f'{at:<8}  n={mask.sum():>9,}  recall={rc:.4f}')

# Bar chart
fig, ax = plt.subplots(figsize=(5,3))
bars = ax.bar(attack_types, recalls, color=['#e05c5c','#f0a060','#5c8ae0'])
ax.bar_label(bars, fmt='%.4f', padding=3)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Recall')
ax.set_title('Per-Type Recall (binary model, f01–f18)')
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'v2_proposed' / 'recall_by_type.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Findings

| Metric | Kim et al. (12 features) | This work (18 features) |
|--------|--------------------------|-------------------------|
| F1     | ~0.98 (reported) / ~0.84 (reproduced) | **~0.9979** |
| DoS recall | — | **~0.9998** |
| Fuzzy recall | — / ~0.34 reproduced | **~0.9991** |
| MITM recall | — | **~0.9972** |

### Critical note on fuzzy recall
The dataset labels **all** packets in the fuzzy PCAP as "fuzzy", including background  
normal traffic. About 83% of packets labelled fuzzy are benign traffic indistinguishable  
from the benign PCAP. The improvement in recall here comes from f17 (`src_packet_rate`),  
which distinguishes the flooding attacker (172.18.0.12/0.17) from normal hosts.

### f17 paradox
f17 shows only ~3.6% feature importance by gain, but removing it drops F1 from 0.9979  
to 0.74. This happens because f17 is a near-perfect binary split for the attacker IPs  
— XGBoost uses it once near the root and delegates the rest to other features.